# 01 · Data Preparation

**Purpose:** Carry forward and modernize data preparation from the legacy notebooks.

**Inputs:** Reference/pbp_head.csv, Reference/fg_attempts_sample.csv

**Outputs:** reports/data_prep_schema_preview.csv

**Sections:**
- [Parameters & Modes](#parameters--modes)
- [Imports](#imports--install-if-missing)
- [Utilities & Helpers](#utilities--helpers)
- [Data Load & Peek](#data-load--peek)
- [Stage Logic](#stage-logic)
- [Artifacts](#artifacts)
- [Session Info](#session-info)


In [2]:
# Parameters & Modes

PROJECT_ROOT <- sub("[/\\\\][^/\\\\]*$", "", getwd())
print(PROJECT_ROOT)
reference_dir <- file.path(PROJECT_ROOT, 'Reference')
pbp_data_dir <- 'C:\\Python\\Data'
data_dir <- file.path(PROJECT_ROOT, 'data')
leverage_dir <- file.path(PROJECT_ROOT,'data','leverage')
reports_dir <- file.path(PROJECT_ROOT, 'reports')
config_path <- file.path(PROJECT_ROOT, 'config', 'params.yaml')

SEASONS <- 2012:2024

default_params <- list(
  time_knots = c(60, 120, 300),
  late_flags = c(120, 60),
  p_clip_min = 0.05,
  p_clip_max = 0.98,
  kickable_cap_modeling = 65,
  kickable_cap_audit = 50,
  tau_grid = c(0.03, 0.05, 0.07, 0.10),
  df_distance = 5,
  df_yardline = 5,
  df_yards_to_go = 5
)

params <- default_params

if (file.exists(config_path)) {
  tryCatch({
    config_values <- yaml::read_yaml(config_path)
    params <- utils::modifyList(params, config_values, keep.null = TRUE)
  }, error = function(e) message('Config read failed, using defaults: ', e$message))
}

print(PROJECT_ROOT)
print(data_dir)
set.seed(20240517)

[1] "g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model"


[1] "g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model"
[1] "g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/data"


In [3]:
# Imports — install if missing

dependencies <- c(
  'dplyr', 'tibble', 'tidyr', 'readr', 'stringr', 'purrr', 'ggplot2',
  'mgcv', 'splines', 'glmmTMB', 'pROC', 'yaml', 'rlang',
  'forcats', 'lubridate', 'nflreadr', 'glue'
 )

installed <- rownames(installed.packages())

for (pkg in dependencies) {
  if (!pkg %in% installed) {
    install.packages(pkg)
  }
  suppressPackageStartupMessages(library(pkg, character.only = TRUE))
}

Warning message:
"package 'glmmTMB' was built under R version 4.5.2"
Warning message:
"package 'pROC' was built under R version 4.5.2"


### Function Index

- `get_schema()` — quick schema preview of a data frame.
- `ensure_columns()` — add defaulted columns when missing.
- `add_time_features()` — derive common time and score features.
- `clip_probabilities()` — constrain probabilities to [min, max].
- `stabilize_weights()` — compute stabilized weights with optional grouping.
- `effective_sample_size()` — calculate ESS from weights.
- `calibration_summary()` — summarize calibration by bins.
- `plot_calibration()` — simple calibration scatter + smoother.


In [ ]:
# # Data Load
# pbp_path <- file.path(pbp_data_dir, 'pbp_2000_2024_clean.csv')
# fg_path <- file.path(fg_data_dir, 'fg_attempts.csv')

# #Load if file exits, if not return error
# if (file.exists(pbp_path)) {
#   pbp <- readr::read_csv(pbp_path, guess_max = 10000, show_col_types = FALSE)
# } else {
#   stop(glue("File {pbp_path} does not exist. Please download the data first."))
# }

# pbp %>%
#   write_rds(file.path(pbp_data_dir, "pbp.rds"))


In [4]:
# Data Load
pbp_path <- file.path(pbp_data_dir, 'pbp.rds')
fg_path <- file.path(data_dir, 'fg_attempts.csv')

pbp <- readr::read_rds(pbp_path)
fg_attempts <- if (file.exists(fg_path)) readr::read_csv(fg_path, show_col_types = FALSE) else NULL

#Filter for 2015 - 2024
pbp <- pbp %>% filter(season %in% SEASONS)


glimpse(pbp)


Rows: 627,226
Columns: 372
$ play_id                              <dbl> 1, 35, 53, 74, 95, 119, 143, 165,…
$ game_id                              <chr> "2012_01_ATL_KC", "2012_01_ATL_KC…
$ old_game_id                          <dbl> 2012090908, 2012090908, 201209090…
$ home_team                            <chr> "KC", "KC", "KC", "KC", "KC", "KC…
$ away_team                            <chr> "ATL", "ATL", "ATL", "ATL", "ATL"…
$ season_type                          <chr> "REG", "REG", "REG", "REG", "REG"…
$ week                                 <dbl> 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, …
$ posteam                              <chr> NA, "ATL", "ATL", "ATL", "ATL", "…
$ posteam_type                         <chr> NA, "away", "away", "away", "away…
$ defteam                              <chr> NA, "KC", "KC", "KC", "KC", "KC",…
$ side_of_field                        <chr> NA, "KC", "ATL", "ATL", "ATL", "A…
$ yardline_100                         <dbl> NA, 35, 80, 74, 72, 69, 44, 44, 4…
$ game_date  

## Previous Play Features (lag within game)

In [5]:
pbp_prev <- pbp %>%
  arrange(game_id, play_id) %>%
  group_by(game_id) %>%
  mutate(
    prev_play_type = dplyr::lag(play_type),
    prev_desc = dplyr::lag(desc),
    prev_timeout = as.integer(dplyr::lag(timeout)),
    prev_timeout_team = dplyr::lag(timeout_team),
    prev_penalty = as.integer(dplyr::lag(penalty)),
    prev_incomplete = as.integer(dplyr::lag(incomplete_pass)),
    prev_out_bounds = as.integer(dplyr::lag(out_of_bounds)),
    prev_gsr = dplyr::lag(game_seconds_remaining),
    delta_secs = prev_gsr - game_seconds_remaining,
    prev_end_quarter = as.integer(!is.na(prev_desc) & str_detect(prev_desc, "(?i)end\\s+quarter")),
    prev_two_min_warning = as.integer(!is.na(prev_desc) & str_detect(prev_desc, "(?i)two-?minute\\s+warning"))
  ) %>%
  ungroup() %>%
  select(
    game_id, play_id,
    prev_play_type, prev_desc, prev_timeout, prev_timeout_team,
    prev_penalty, prev_incomplete, prev_out_bounds,
    prev_end_quarter, prev_two_min_warning, delta_secs
)

print(head(pbp_prev))

# A tibble: 6 × 12
  game_id        play_id prev_play_type prev_desc prev_timeout prev_timeout_team
  <chr>            <dbl> <chr>          <chr>            <int> <chr>            
1 2012_01_ATL_KC       1 NA             NA                  NA NA               
2 2012_01_ATL_KC      35 NA             GAME                NA NA               
3 2012_01_ATL_KC      53 kickoff        6-R.Succ…            0 NA               
4 2012_01_ATL_KC      74 run            (15:00) …            0 NA               
5 2012_01_ATL_KC      95 run            (14:22) …            0 NA               
6 2012_01_ATL_KC     119 pass           (13:41) …            0 NA               
# ℹ 6 more variables: prev_penalty <int>, prev_incomplete <int>,
#   prev_out_bounds <int>, prev_end_quarter <int>, prev_two_min_warning <int>,
#   delta_secs <dbl>


## Field Goal & PAT Attempts

In [6]:
fg_attempts_raw <- pbp %>%
  filter(
    season %in% SEASONS,
    play_type %in% c("field_goal", "extra_point"),
    (field_goal_result %in% c("made", "missed", "blocked")) |
      (extra_point_result %in% c("good", "failed", "blocked"))
  ) %>%
  mutate(
    is_pat = as.integer(play_type == "extra_point"),
    kick_result_raw = dplyr::coalesce(field_goal_result, extra_point_result),
    kick_result = case_when(
      kick_result_raw %in% c("made", "good") ~ "made",
      kick_result_raw %in% c("missed", "failed") ~ "missed",
      kick_result_raw %in% c("blocked") ~ "blocked",
      TRUE ~ NA_character_
    ),
    kick_distance = suppressWarnings(as.numeric(kick_distance)),
    kick_distance = if_else(is_pat == 1L & is.na(kick_distance), 33, kick_distance),
    attempted = 1L
  ) %>%
  filter(!is.na(kick_distance)) %>%
  transmute(
    game_id, play_id, old_game_id,
    season = as.integer(season),
    week = as.integer(week),
    season_type,
    playoffs = as.integer(season_type == "POST"),
    qtr = as.integer(qtr),
    game_date = as.Date(game_date),
    home_team, away_team, posteam, defteam,
    game_seconds_remaining, quarter_seconds_remaining,
    score_differential, yardline_100, ydstogo,
    wp, wpa, epa,
    posteam_timeouts_remaining, defteam_timeouts_remaining,
    goal_to_go,
    is_ot  = qtr >= 5,
    roof, surface, temp, wind, weather,
    stadium_id, stadium, location,
    kick_distance, kicker_player_id, kicker_player_name,
    field_goal_result, extra_point_result,
    kick_result, is_pat, attempted,
    play_type_original = play_type
  )

print(head(fg_attempts_raw))

# A tibble: 6 × 42
  game_id play_id old_game_id season  week season_type playoffs   qtr game_date 
  <chr>     <dbl>       <dbl>  <int> <int> <chr>          <int> <int> <date>    
1 2012_0…     321  2012090908   2012     1 REG                0     1 2012-09-09
2 2012_0…     588  2012090908   2012     1 REG                0     1 2012-09-09
3 2012_0…     727  2012090908   2012     1 REG                0     1 2012-09-09
4 2012_0…     983  2012090908   2012     1 REG                0     2 2012-09-09
5 2012_0…    1213  2012090908   2012     1 REG                0     2 2012-09-09
6 2012_0…    1427  2012090908   2012     1 REG                0     2 2012-09-09
# ℹ 33 more variables: home_team <chr>, away_team <chr>, posteam <chr>,
#   defteam <chr>, game_seconds_remaining <dbl>,
#   quarter_seconds_remaining <dbl>, score_differential <dbl>,
#   yardline_100 <dbl>, ydstogo <dbl>, wp <dbl>, wpa <dbl>, epa <dbl>,
#   posteam_timeouts_remaining <dbl>, defteam_timeouts_remaining <dbl>,
#   go

## Fourth-Down Non-Attempt Opportunities

In [7]:
fg_nonattempts_raw <- pbp %>%
  filter(
    season %in% SEASONS,
    down == 4L,
    !is.na(yardline_100),
    !is.na(ydstogo)
  ) %>%
  mutate(
    derived_kick_distance = yardline_100 + 17L,
  ) %>%
  filter(
    play_type %in% c("pass","run","punt","qb_kneel","qb_spike") |
      (special_teams_play == 1 & field_goal_attempt == 0 &
         play_type %in% c("pass","run"))
  ) %>%
  transmute(
    game_id, play_id, old_game_id,
    season = as.integer(season),
    week = as.integer(week),
    season_type,
    playoffs = as.integer(season_type == "POST"),
    qtr = as.integer(qtr), down,
    game_date = as.Date(game_date),
    home_team, away_team, posteam, defteam,
    game_seconds_remaining, quarter_seconds_remaining,
    score_differential, yardline_100, ydstogo,
    wp, wpa, epa,
    posteam_timeouts_remaining, defteam_timeouts_remaining,
    goal_to_go,
    roof, surface, temp, wind, weather,
    stadium_id, stadium, location,
    kick_distance = derived_kick_distance,
    kicker_player_id = NA_character_,
    kicker_player_name = NA_character_,
    field_goal_result = NA_character_,
    extra_point_result = NA_character_,
    kick_result = NA_character_,
    is_pat = 0L,
    is_ot  = qtr >= 5,
    attempted = 0L,
    play_type_original = play_type
  )
print(head(fg_nonattempts_raw))

# A tibble: 6 × 43
  game_id      play_id old_game_id season  week season_type playoffs   qtr  down
  <chr>          <dbl>       <dbl>  <int> <int> <chr>          <int> <int> <dbl>
1 2012_01_ATL…    3124  2012090908   2012     1 REG                0     4     4
2 2012_01_ATL…    3263  2012090908   2012     1 REG                0     4     4
3 2012_01_BUF…    1110  2012090902   2012     1 REG                0     2     4
4 2012_01_BUF…    2340  2012090902   2012     1 REG                0     3     4
5 2012_01_BUF…    3042  2012090902   2012     1 REG                0     4     4
6 2012_01_BUF…    3620  2012090902   2012     1 REG                0     4     4
# ℹ 34 more variables: game_date <date>, home_team <chr>, away_team <chr>,
#   posteam <chr>, defteam <chr>, game_seconds_remaining <dbl>,
#   quarter_seconds_remaining <dbl>, score_differential <dbl>,
#   yardline_100 <dbl>, ydstogo <dbl>, wp <dbl>, wpa <dbl>, epa <dbl>,
#   posteam_timeouts_remaining <dbl>, defteam_timeouts_remai

In [8]:
# ensure output directory exists and write with basic error handling
if (!dir.exists(data_dir)) dir.create(data_dir, recursive = TRUE, showWarnings = FALSE)

out_path <- file.path(data_dir, 'fg_nonattempts_raw.csv')
tryCatch({
  readr::write_csv(fg_nonattempts_raw, out_path)
  message("Wrote: ", out_path)
}, error = function(e) {
  stop("Failed to write ", out_path, ": ", e$message, call. = FALSE)
})

Wrote: g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/data/fg_nonattempts_raw.csv



In [9]:
pbp %>%
  filter(season %in% SEASONS, qtr == 4L, down == 4L,
         !is.na(yardline_100), !is.na(ydstogo)) %>%
  mutate(dist = yardline_100 + 17L) %>%
  filter(dplyr::between(score_differential, -10, 6),
         ydstogo >= 1, dist <= 50) %>%
  count(play_type, sort = TRUE)

play_type,n
<chr>,<int>
field_goal,1662
pass,387
run,187
no_play,126
punt,1
qb_kneel,1
NA,1


In [10]:
pbp %>%
  filter(season %in% SEASONS, qtr == 4L, down == 4L,
         !is.na(yardline_100), !is.na(ydstogo)) %>%
  mutate(dist = yardline_100 + 17L,
         in_window = dplyr::between(score_differential, -10, 6) &
                     ydstogo >= 1 & dist <= 50) %>%
  summarise(
    total_window = sum(in_window),
    kicks = sum(in_window & play_type == 'field_goal'),
    non_kicks = sum(in_window &
                    (play_type %in% c('pass','run','punt','qb_kneel','qb_spike') |
                       (special_teams_play == 1 & field_goal_attempt == 0 &
                          play_type %in% c('pass','run'))))
  )

total_window,kicks,non_kicks
<int>,<int>,<int>
2365,NA,576


## Combine Attempts & Opportunities for Shared Feature Engineering

In [11]:
fg_all <- bind_rows(fg_attempts_raw, fg_nonattempts_raw) %>%
  left_join(pbp_prev, by = c('game_id', 'play_id'))

## Situational Features

In [12]:
fg_all <- fg_all %>%
  mutate(
    l2m = as.integer(qtr %in% c(2L, 4L) & quarter_seconds_remaining <= 120),
    clock_running = as.integer(
      is_pat == 0L &
      !is.na(delta_secs) & delta_secs > 0 &
      coalesce(prev_timeout, 0L) == 0L &
      coalesce(prev_penalty, 0L) == 0L &
      coalesce(prev_incomplete, 0L) == 0L &
      coalesce(prev_out_bounds, 0L) == 0L &
      coalesce(prev_end_quarter, 0L) == 0L &
      coalesce(prev_two_min_warning, 0L) == 0L
    ),
    iced = as.integer(
      is_pat == 0L &
      coalesce(prev_timeout, 0L) == 1L &
      !is.na(prev_timeout_team) &
      prev_timeout_team == defteam
    )
  )

## Kicker Information (age & experience)

In [13]:
players <- nflreadr::load_players() %>%
  transmute(
    gsis_id = as.character(gsis_id),
    birth_date = as.Date(birth_date),
    rookie_season = suppressWarnings(as.integer(rookie_season))
  )

fg_all <- fg_all %>%
  mutate(kicker_player_id = as.character(kicker_player_id)) %>%
  left_join(players, by = c('kicker_player_id' = 'gsis_id')) %>%
  mutate(
    kicker_age = as.numeric(difftime(game_date, birth_date, units = 'days')) / 365.25,
    kicker_experience = if_else(
      !is.na(rookie_season),
      as.numeric(season - rookie_season) + 1,
      NA_real_
    )
  )

## Venue Flags

In [14]:
fg_all <- fg_all %>%
  mutate(
    roof = forcats::fct_explicit_na(as.factor(roof), 'unknown'),
    surface = forcats::fct_explicit_na(as.factor(surface), 'unknown'),
    roof_std = tolower(as.character(roof)),
    indoors = as.integer(roof_std %in% c('dome', 'closed', 'open')),
    surface_std = tolower(as.character(surface)),
    is_turf = as.integer(surface_std != 'grass'),
    high_altitude = as.integer(!is.na(stadium_id) & str_detect(stadium_id, '^(DEN|MEX)'))
  )

Warning message:
"There was 1 warning in `mutate()`.
ℹ In argument: `roof = forcats::fct_explicit_na(as.factor(roof), "unknown")`.
Caused by warning:
! `fct_explicit_na()` was deprecated in forcats 1.0.0.
ℹ Please use `fct_na_value_to_level()` instead."


## Weather Parsing & Imputation

In [15]:
fg_all <- fg_all %>%
  mutate(
    weather = if_else(weather == '', NA_character_, weather),
    weather_first = if_else(
      is.na(weather),
      NA_character_,
      str_squish(str_to_lower(str_replace(weather, '(?i)\\s*temp:.*$', '')))
    ),
    weather_clean = weather_first,
    temp_from_weather = str_extract(weather, '(?i)temp\\s*:?\\s*(-?\\d{1,3})'),
    temp_from_weather = suppressWarnings(as.numeric(str_extract(temp_from_weather, '-?\\d{1,3}'))),
    temp_degree = str_extract(weather, '(?i)(-?\\d{1,3})\\s*(?:deg|degrees|\\u00B0)?\\s*f'),
    temp_degree = suppressWarnings(as.numeric(str_extract(temp_degree, '-?\\d{1,3}'))),
    wind_from_weather = str_extract(weather, '(?i)(\\d{1,3})\\s*mph'),
    wind_from_weather = suppressWarnings(as.numeric(str_extract(wind_from_weather, '\\d{1,3}'))),
    humidity_from_weather = str_extract(weather, '(?i)(\\d{1,3})\\s*%'),
    humidity_from_weather = suppressWarnings(as.numeric(str_extract(humidity_from_weather, '\\d{1,3}'))),
    temp = coalesce(temp, temp_from_weather, temp_degree),
    wind = coalesce(wind, wind_from_weather),
    humidity = humidity_from_weather
  ) %>%
  mutate(
    temp = if_else(is.na(temp) & indoors == 1L, 71, temp),
    wind = if_else(is.na(wind) & indoors == 1L, 0, wind),
    humidity = if_else(is.na(humidity) & indoors == 1L, 45, humidity)
  )

outdoor_monthly_medians <- fg_all %>%
  filter(indoors == 0L) %>%
  mutate(month = lubridate::month(game_date)) %>%
  group_by(stadium_id, month) %>%
  summarise(
    temp_median = suppressWarnings(as.numeric(median(temp, na.rm = TRUE))),
    wind_median = suppressWarnings(as.numeric(median(wind, na.rm = TRUE))),
    humidity_median = suppressWarnings(as.numeric(median(humidity, na.rm = TRUE))),
    .groups = 'drop'
  )


fg_all <- fg_all %>%
  mutate(month = lubridate::month(game_date)) %>%
  left_join(outdoor_monthly_medians, by = c('stadium_id', 'month')) %>%
  mutate(
    temp = if_else(is.na(temp) & indoors == 0L, temp_median, temp),
    wind = if_else(is.na(wind) & indoors == 0L, wind_median, wind),
    humidity = if_else(is.na(humidity) & indoors == 0L, humidity_median, humidity)
  )

global_temp_median <- fg_all %>% filter(indoors == 0L, !is.na(temp)) %>% summarise(median = median(temp)) %>% pull()
if (length(global_temp_median) == 0) global_temp_median <- NA_real_

global_wind_median <- fg_all %>% filter(indoors == 0L, !is.na(wind)) %>% summarise(median = median(wind)) %>% pull()
if (length(global_wind_median) == 0) global_wind_median <- NA_real_

global_humidity_median <- fg_all %>% filter(indoors == 0L, !is.na(humidity)) %>% summarise(median = median(humidity)) %>% pull()
if (length(global_humidity_median) == 0) global_humidity_median <- NA_real_

fg_all <- fg_all %>%
  mutate(
    temp = if_else(is.na(temp) & indoors == 0L, global_temp_median, temp),
    wind = if_else(is.na(wind) & indoors == 0L, global_wind_median, wind),
    humidity = if_else(is.na(humidity) & indoors == 0L, global_humidity_median, humidity)
  ) %>%
  # Final logic check for indoor venues:
  mutate(
    # ensure roof_std exists and is lowercase; fallback to tolower(roof) if needed
    roof_std = if_else(is.na(roof_std), tolower(as.character(roof)), roof_std),
    temp = case_when(
      indoors == 1L & roof_std %in% c('dome', 'closed') ~ 71,
      TRUE ~ temp
    ),
    wind = case_when(
      indoors == 1L & roof_std %in% c('dome', 'closed') ~ 0,
      indoors == 1L & roof_std == 'open' & !is.na(wind) & wind > 5 ~ 5,
      TRUE ~ wind
    )
  ) %>%
  select(-temp_from_weather, -temp_degree, -wind_from_weather, -humidity_from_weather,
         -temp_median, -wind_median, -humidity_median, -month)

## Weather Flags

In [16]:
fg_all <- fg_all %>%
  mutate(
    weather_clean = str_to_lower(coalesce(weather, '')) %>% str_squish(),
    weather_clean = if_else(weather_clean == '' & indoors == 0L, 'clear', weather_clean),
    weather_clean = str_replace_all(weather_clean, '(cloudly|coudy|cloundy|clo[iu]dy)', 'cloudy'),
    weather_clean = str_replace_all(weather_clean, '(mosly|mostly\\s+coudy)', 'mostly cloudy'),
    weather_clean = str_replace_all(weather_clean, '(partly\\s*sunny|sun\\s*/\\s*clouds|sun\\s*&\\s*clouds|sunny\\s*intervals)', 'partly cloudy'),
    weather_clean = str_replace_all(weather_clean, 'hazey', 'hazy'),
    no_rain_phrase = as.integer(str_detect(weather_clean, 'no\\s+chance\\s+of\\s+rain|0%\\s*chance\\s*of\\s+rain|zero\\s*percent\\s*chance\\s*of\\s+rain')),
    is_snow_sleet = as.integer(
      indoors != 1L & str_detect(weather_clean, '\\bsnow\\b|\\bflurr(y|ies)\\b|\\bsleet\\b|\\bfreezing\\s+rain\\b|\\bwintry\\s+mix\\b|\\bice\\b')
    ),
    is_rain_showers = as.integer(
      indoors != 1L & str_detect(weather_clean, '\\brain\\b|\\braining\\b|\\bshowers?\\b|\\bdrizzle\\b|\\bstorm\\b|\\bthunderstorm\\b')
    ),
    is_rain_showers = if_else(is_snow_sleet == 1L | no_rain_phrase == 1L, 0L, is_rain_showers),
    is_cloudy = as.integer(
      indoors != 1L & str_detect(weather_clean, '\\bcloudy\\b|\\bovercast\\b|\\bpartly\\s+cloudy\\b|\\bmostly\\s+cloudy\\b|\\bscattered\\s+clouds?\\b')
    ),
    is_hazy_fog = as.integer(
      indoors != 1L & str_detect(weather_clean, '\\bfog(?:gy)?\\b|\\bhaze\\b|\\bhazy\\b|\\bmist\\b')
    )
  ) %>%
  select(-no_rain_phrase)

## Adding Leverage

In [17]:
leverage_pack_ok <- TRUE
leverage_dir <- file.path(PROJECT_ROOT, 'data', 'leverage')
fg_pack_path  <- file.path(leverage_dir, 'wp_heads_rulepack.rds')
pat_pack_path <- file.path(leverage_dir, 'pat_heads_rulepack.rds')

if (!file.exists(fg_pack_path) || !file.exists(pat_pack_path)) {
  leverage_pack_ok <- FALSE
  warning('Leverage rulepacks missing; skipping leverage augmentation. Expected in ', leverage_dir)
}

if (leverage_pack_ok) {
  fg_pack <- readRDS(fg_pack_path)
  pat_pack <- readRDS(pat_pack_path)

  mod_make        <- fg_pack$mod_make
  mod_miss        <- fg_pack$mod_miss
  mk_feats_make   <- fg_pack$mk_feats_make
  mk_feats_miss   <- fg_pack$mk_feats_miss
  apply_end_rules <- fg_pack$apply_endgame_rules

  mod_pat_make      <- pat_pack$mod_pat_make
  mod_pat_miss      <- pat_pack$mod_pat_miss
  mk_feats_pat_make <- pat_pack$mk_feats_pat_make
  mk_feats_pat_miss <- pat_pack$mk_feats_pat_miss
  apply_pat_rules   <- pat_pack$apply_pat_rules_min
  pat_ot_const      <- pat_pack$pat_ot_const

  eps <- 1e-6
  clamp01 <- function(x) pmin(pmax(x, eps), 1 - eps)
  desired_cols <- c(
    '.row_id',
    'wp_make_hat', 'wp_miss_hat', 'leverage',
    'wp_make_hat_rules', 'wp_miss_hat_rules',
    'pred_hat_post',
    'leverage_rules'
  )

  fg_all <- fg_all %>% mutate(.row_id = dplyr::row_number())

  fg_nonpat <- fg_all %>% filter(is_pat == 0L)
  fg_nonpat_scorable <- fg_nonpat %>%
    filter(!is_ot) %>%
    filter(!is.na(yardline_100))

  if (nrow(fg_nonpat_scorable) > 0) {
    feats_make <- mk_feats_make(fg_nonpat_scorable)
    feats_miss <- mk_feats_miss(fg_nonpat_scorable)

    scored_fg <- fg_nonpat_scorable %>%
      mutate(
        wp_make_hat = clamp01(predict(mod_make, newdata = feats_make, type = 'response')),
        wp_miss_hat = clamp01(predict(mod_miss, newdata = feats_miss, type = 'response')),
        leverage    = pmin(pmax(wp_make_hat - wp_miss_hat, 0), 1)
      ) %>%
      apply_end_rules() %>%
      dplyr::select(dplyr::any_of(desired_cols))
  } else {
    scored_fg <- tibble(.row_id = integer(),
                        wp_make_hat = numeric(), wp_miss_hat = numeric(), leverage = numeric(),
                        wp_make_hat_rules = numeric(), wp_miss_hat_rules = numeric(),
                        pred_hat_post = numeric(), leverage_rules = numeric())
  }

  fg_nonpat_ot <- fg_nonpat %>% filter(is_ot)

  const_pool <- fg_nonpat_scorable %>%
    inner_join(
      scored_fg %>% dplyr::select(.row_id, leverage_rules),
      by = '.row_id'
    ) %>%
    dplyr::filter(
      qtr == 4L,
      quarter_seconds_remaining < 120,
      score_differential %in% -3:0
    )

  fg_ot_const_val <- const_pool %>%
    summarise(med = median(leverage_rules, na.rm = TRUE)) %>%
    pull(med)

  if (is.na(fg_ot_const_val)) {
    fg_ot_const_val <- scored_fg %>%
      summarise(med = median(leverage_rules, na.rm = TRUE)) %>% pull(med)
  }
  if (is.na(fg_ot_const_val)) fg_ot_const_val <- 0.5

  if (nrow(fg_nonpat_ot) > 0) {
    scored_fg_ot <- fg_nonpat_ot %>%
      transmute(
        .row_id,
        wp_make_hat       = NA_real_,
        wp_miss_hat       = NA_real_,
        leverage          = fg_ot_const_val,
        wp_make_hat_rules = NA_real_,
        wp_miss_hat_rules = NA_real_,
        pred_hat_post     = NA_real_,
        leverage_rules    = fg_ot_const_val
      ) %>%
      dplyr::select(dplyr::any_of(desired_cols))
  } else {
    scored_fg_ot <- tibble(.row_id = integer(),
                           wp_make_hat = numeric(), wp_miss_hat = numeric(), leverage = numeric(),
                           wp_make_hat_rules = numeric(), wp_miss_hat_rules = numeric(),
                           pred_hat_post = numeric(), leverage_rules = numeric())
  }

  pats <- fg_all %>% filter(is_pat == 1L)
  pats_nonot <- pats %>% filter(!is_ot)
  pats_ot    <- pats %>% filter(is_ot)

  if (nrow(pats_nonot) > 0) {
    feats_pat_make <- mk_feats_pat_make(pats_nonot)
    feats_pat_miss <- mk_feats_pat_miss(pats_nonot)

    scored_pat_nonot <- pats_nonot %>%
      mutate(
        wp_make_hat = clamp01(predict(mod_pat_make, newdata = feats_pat_make, type = 'response')),
        wp_miss_hat = clamp01(predict(mod_pat_miss, newdata = feats_pat_miss, type = 'response')),
        leverage    = pmin(pmax(wp_make_hat - wp_miss_hat, 0), 1)
      ) %>%
      apply_pat_rules() %>%
      dplyr::select(dplyr::any_of(desired_cols))
  } else {
    scored_pat_nonot <- tibble(.row_id = integer(),
                               wp_make_hat = numeric(), wp_miss_hat = numeric(), leverage = numeric(),
                               wp_make_hat_rules = numeric(), wp_miss_hat_rules = numeric(),
                               pred_hat_post = numeric(), leverage_rules = numeric())
  }

  if (nrow(pats_ot) > 0) {
    scored_pat_ot <- pat_ot_const(pats_ot) %>%
      dplyr::select(dplyr::any_of(desired_cols))
  } else {
    scored_pat_ot <- tibble(.row_id = integer(),
                            wp_make_hat = numeric(), wp_miss_hat = numeric(), leverage = numeric(),
                            wp_make_hat_rules = numeric(), wp_miss_hat_rules = numeric(),
                            pred_hat_post = numeric(), leverage_rules = numeric())
  }

  scored_all <- bind_rows(scored_fg, scored_fg_ot, scored_pat_nonot, scored_pat_ot)

  fg_all <- fg_all %>%
    left_join(scored_all, by = '.row_id') %>%
    select(-.row_id)

  message(sprintf(
    'Leverage added: FG (reg) = %d, FG (OT const) = %d, PAT (reg) = %d, PAT (OT) = %d. FG OT const = %.4f',
    nrow(scored_fg), nrow(scored_fg_ot), nrow(scored_pat_nonot), nrow(scored_pat_ot), fg_ot_const_val
  ))

  fg_made_vals   <- c('made', 'good')
  fg_miss_vals   <- c('missed', 'blocked')
  pat_good_vals  <- c('good')
  pat_fail_vals  <- c('failed', 'missed', 'blocked', 'aborted', 'no_good')

  fg_all <- fg_all %>%
    mutate(
      pred_hat_post = dplyr::case_when(
        is_pat == 0L & attempted == 1L & !is.na(kick_result) &
          tolower(kick_result) %in% fg_made_vals ~ wp_make_hat_rules,

        is_pat == 0L & attempted == 1L & !is.na(kick_result) &
          tolower(kick_result) %in% fg_miss_vals ~ wp_miss_hat_rules,

        is_pat == 1L & !is.na(extra_point_result) &
          tolower(extra_point_result) %in% pat_good_vals ~ wp_make_hat_rules,

        is_pat == 1L & !is.na(extra_point_result) &
          tolower(extra_point_result) %in% pat_fail_vals ~ wp_miss_hat_rules,

        TRUE ~ pred_hat_post
      ),
      pred_hat_post = dplyr::if_else(
        !is.na(pred_hat_post),
        pmin(pmax(pred_hat_post, 0), 1),
        pred_hat_post
      )
    )
} else {
  leverage_cols <- c('wp_make_hat','wp_miss_hat','leverage','wp_make_hat_rules','wp_miss_hat_rules','leverage_rules','pred_hat_post')
  for (col in leverage_cols) {
    if (!col %in% names(fg_all)) {
      fg_all[[col]] <- NA_real_
    }
  }
}

Leverage added: FG (reg) = 52090, FG (OT const) = 431, PAT (reg) = 16656, PAT (OT) = 0. FG OT const = 0.4936



## Sanity Checks & Completeness

In [18]:
suppressPackageStartupMessages({
  library(glue)
})

assert_true <- function(ok, msg) if (!isTRUE(ok)) stop(msg, call. = FALSE)
assert_between01 <- function(x, allow_na = TRUE, label = deparse(substitute(x))) {
  if (allow_na) x <- x[!is.na(x)]
  bad <- any(x < 0 | x > 1)
  assert_true(!bad, glue('{label} must be within [0,1].'))
}

fg_chk <- fg_all %>% mutate(.row_id_check = dplyr::row_number())

fg_chk <- fg_chk %>%
  mutate(
    pat_flag = as.logical(is_pat),
    ot_flag  = as.logical(is_ot)
  )

req_cols <- c(
  'is_pat', 'is_ot',
  'wp_make_hat', 'wp_miss_hat', 'leverage',
  'wp_make_hat_rules', 'wp_miss_hat_rules', 'leverage_rules'
 )
missing_cols <- setdiff(req_cols, names(fg_chk))
assert_true(length(missing_cols) == 0,
            glue('Missing expected columns: {paste(missing_cols, collapse=", ")}'))

assert_between01(fg_chk$wp_make_hat,       TRUE, 'wp_make_hat')
assert_between01(fg_chk$wp_miss_hat,       TRUE, 'wp_miss_hat')
assert_between01(fg_chk$leverage,          TRUE, 'leverage')
assert_between01(fg_chk$wp_make_hat_rules, TRUE, 'wp_make_hat_rules')
assert_between01(fg_chk$wp_miss_hat_rules, TRUE, 'wp_miss_hat_rules')
assert_between01(fg_chk$leverage_rules,    TRUE, 'leverage_rules')

fg_chk <- fg_chk %>%
  mutate(.lev_raw_from_heads = pmin(pmax(wp_make_hat - wp_miss_hat, 0), 1))
diff_eps <- with(fg_chk, abs(leverage - .lev_raw_from_heads))
if (any(!is.na(diff_eps))) {
  assert_true(max(diff_eps, na.rm = TRUE) < 1e-6,
              'leverage != clamp(wp_make_hat - wp_miss_hat) on some rows.')
}

fg_nonpat <- fg_chk %>% filter(!pat_flag)
fg_pats   <- fg_chk %>% filter( pat_flag)

fg_nonpat_nonot <- fg_nonpat %>% filter(!ot_flag)
fg_nonpat_nonot_na <- fg_nonpat_nonot %>%
  summarise(
    n = n(),
    na_wp_make = sum(is.na(wp_make_hat)),
    na_wp_miss = sum(is.na(wp_miss_hat)),
    na_lev     = sum(is.na(leverage))
  )

fg_nonpat_ot <- fg_nonpat %>% filter(ot_flag)

pats_na <- fg_pats %>%
  summarise(
    n = n(),
    na_wp_make    = sum(is.na(wp_make_hat)),
    na_wp_miss    = sum(is.na(wp_miss_hat)),
    na_lev        = sum(is.na(leverage)),
    na_lev_rules  = sum(is.na(leverage_rules))
  )
assert_true(all(pats_na[1, c('na_wp_make','na_wp_miss','na_lev','na_lev_rules')] == 0),
            'PAT rows contain NAs but should be fully scored (including OT).')

fg_nonpat_nonot_rules_na <- fg_nonpat_nonot %>% summarise(na_lev_rules = sum(is.na(leverage_rules)))
assert_true(fg_nonpat_nonot_rules_na$na_lev_rules == 0,
            'Non-OT FG rows have NA in leverage_rules unexpectedly.')

summary_counts <- fg_chk %>%
  mutate(kind = if_else(pat_flag, 'PAT', 'FG'),
         period = if_else(ot_flag, 'OT', 'Reg')) %>%
  count(kind, period, name = 'plays')

na_summary <- fg_chk %>%
  mutate(kind = if_else(pat_flag, 'PAT', 'FG'),
         period = if_else(ot_flag, 'OT', 'Reg')) %>%
  summarise(
    plays        = n(),
    wp_make_na   = sum(is.na(wp_make_hat)),
    wp_miss_na   = sum(is.na(wp_miss_hat)),
    lev_na       = sum(is.na(leverage)),
    lev_rules_na = sum(is.na(leverage_rules)),
    .by = c(kind, period)
  ) %>%
  arrange(kind, period)

range_summary <- fg_chk %>%
  summarise(
    across(
      c(wp_make_hat, wp_miss_hat, leverage, wp_make_hat_rules, wp_miss_hat_rules, leverage_rules),
      list(min = ~min(.x, na.rm = TRUE), max = ~max(.x, na.rm = TRUE)),
      .names = '{.col}_{.fn}'
    )
  )

cat('\n=== Sanity Check Report =====================================\n')
summary_counts
cat('\n--- NA counts by group ---------------------------------------\n')
na_summary
cat('\n--- Value ranges (excluding NAs) ------------------------------\n')
range_summary

edge_hi <- fg_chk %>%
  filter(!is.na(leverage_rules)) %>%
  arrange(desc(leverage_rules)) %>%
  select(game_id, play_id, is_pat, is_ot, wp_make_hat_rules, wp_miss_hat_rules, leverage_rules) %>%
  head(5)

edge_lo <- fg_chk %>%
  filter(!is.na(leverage_rules)) %>%
  arrange(leverage_rules) %>%
  select(game_id, play_id, is_pat, is_ot, wp_make_hat_rules, wp_miss_hat_rules, leverage_rules) %>%
  head(5)

cat('\n--- Top 5 highest leverage_rules ------------------------------\n'); edge_hi
cat('\n--- Top 5 lowest leverage_rules -------------------------------\n'); edge_lo
cat('\nSanity checks completed.\n')


=== Sanity Check Report =====================================


kind,period,plays
<chr>,<chr>,<int>
FG,OT,431
FG,Reg,52090
PAT,Reg,16656



--- NA counts by group ---------------------------------------


kind,period,plays,wp_make_na,wp_miss_na,lev_na,lev_rules_na
<chr>,<chr>,<int>,<int>,<int>,<int>,<int>
FG,OT,431,431,431,0,0
FG,Reg,52090,0,0,0,0
PAT,Reg,16656,0,0,0,0



--- Value ranges (excluding NAs) ------------------------------


...


--- Top 5 highest leverage_rules ------------------------------


game_id,play_id,is_pat,is_ot,wp_make_hat_rules,wp_miss_hat_rules,leverage_rules
<chr>,<dbl>,<int>,<lgl>,<dbl>,<dbl>,<dbl>
2012_02_ARI_NE,4435,0,FALSE,0.995,0.005,0.99
2012_03_NE_BAL,4771,0,FALSE,0.995,0.005,0.99
2012_05_PHI_PIT,4153,0,FALSE,0.995,0.005,0.99
2012_08_CAR_CHI,4312,0,FALSE,0.995,0.005,0.99
2012_14_DAL_CIN,4230,0,FALSE,0.995,0.005,0.99



--- Top 5 lowest leverage_rules -------------------------------


game_id,play_id,is_pat,is_ot,wp_make_hat_rules,wp_miss_hat_rules,leverage_rules
<chr>,<dbl>,<int>,<lgl>,<dbl>,<dbl>,<dbl>
2012_05_BUF_SF,3698,1,FALSE,0.9999990,0.9999990,0
2012_09_CHI_TEN,1176,1,FALSE,0.9596418,0.9597301,0
2012_14_ARI_SEA,3876,1,FALSE,0.9999990,0.9999990,0
2012_16_TEN_GB,3097,1,FALSE,0.9998017,0.9998026,0
2012_16_TEN_GB,3253,1,FALSE,0.9999704,0.9999714,0



Sanity checks completed.


## Standardized Features & Modeling Flags

In [19]:
stats_attempts <- fg_all %>%
  filter(attempted == 1L) %>%
  summarise(
    wind_mean = mean(wind, na.rm = TRUE),
    wind_sd = sd(wind, na.rm = TRUE),
    temp_mean = mean(temp, na.rm = TRUE),
    temp_sd = sd(temp, na.rm = TRUE),
    humidity_mean = mean(humidity, na.rm = TRUE),
    humidity_sd = sd(humidity, na.rm = TRUE),
    age_mean = mean(kicker_age, na.rm = TRUE),
    age_sd = sd(kicker_age, na.rm = TRUE),
    exp_mean = mean(kicker_experience, na.rm = TRUE),
    exp_sd = sd(kicker_experience, na.rm = TRUE),
    season_mean = mean(season, na.rm = TRUE),
    season_sd = sd(season, na.rm = TRUE)
  )

wind_mean <- stats_attempts$wind_mean
wind_sd <- stats_attempts$wind_sd
if (is.na(wind_sd) || wind_sd == 0) wind_sd <- NA_real_

temp_mean <- stats_attempts$temp_mean
temp_sd <- stats_attempts$temp_sd
if (is.na(temp_sd) || temp_sd == 0) temp_sd <- NA_real_

humidity_mean <- stats_attempts$humidity_mean
humidity_sd <- stats_attempts$humidity_sd
if (is.na(humidity_sd) || humidity_sd == 0) humidity_sd <- NA_real_

age_mean <- stats_attempts$age_mean
age_sd <- stats_attempts$age_sd
if (is.na(age_sd) || age_sd == 0) age_sd <- NA_real_

exp_mean <- stats_attempts$exp_mean
exp_sd <- stats_attempts$exp_sd
if (is.na(exp_sd) || exp_sd == 0) exp_sd <- NA_real_

season_mean <- stats_attempts$season_mean
season_sd <- stats_attempts$season_sd
if (is.na(season_sd) || season_sd == 0) season_sd <- NA_real_

lev_mean <- mean(fg_all$leverage_rules, na.rm = TRUE)
lev_sd   <- stats::sd(fg_all$leverage_rules, na.rm = TRUE)

fg_all <- fg_all %>%
  mutate(
    wind_z = ifelse(!is.na(wind) & !is.na(wind_sd), (wind - wind_mean) / wind_sd, NA_real_),
    temp_z = ifelse(!is.na(temp) & !is.na(temp_sd), (temp - temp_mean) / temp_sd, NA_real_),
    humidity_z = ifelse(!is.na(humidity) & !is.na(humidity_sd), (humidity - humidity_mean) / humidity_sd, NA_real_),
    kicker_age_z = ifelse(!is.na(kicker_age) & !is.na(age_sd), (kicker_age - age_mean) / age_sd, NA_real_),
    kicker_experience_z = ifelse(!is.na(kicker_experience) & !is.na(exp_sd), (kicker_experience - exp_mean) / exp_sd, NA_real_),
    season_z = ifelse(!is.na(season) & !is.na(season_sd), (season - season_mean) / season_sd, NA_real_),
    kick_made = ifelse(attempted == 1L & !is.na(kick_result), as.integer(kick_result == 'made'), NA_integer_),
    eoh_urgency = as.integer(qtr == 2L & quarter_seconds_remaining <= 10 & clock_running == 1L),
    eog_urgency = as.integer(qtr == 4L & l2m == 1L & clock_running == 1L & !is.na(score_differential) & abs(score_differential) <= 3),
    go_ahead = as.integer(score_differential == 0),
    to_tie = as.integer(score_differential == -3),
    one_score = as.integer(!is.na(score_differential) & abs(score_differential) <= 8),
    leverage_z = ifelse(!is.na(leverage_rules) & is.finite(lev_sd) & lev_sd > 0,
                        (leverage_rules - lev_mean) / lev_sd, NA_real_)
  )

## Split Data & Basic QA

In [20]:
fg_attempts <- fg_all %>% filter(attempted == 1L) %>% arrange(season, week, game_id, play_id)
fg_nonattempts <- fg_all %>% filter(attempted == 0L) %>% arrange(season, week, game_id, play_id)

fg_attempts %>%
  select(game_id, season, week, kick_result, kick_distance, wind, wind_z, temp, temp_z, kicker_player_name, kick_made) %>%
  head()

fg_nonattempts %>%
  select(game_id, season, week, posteam, defteam, kick_distance, wind, temp, ydstogo, score_differential) %>%
  head()

fg_attempts %>%
  summarise(
    attempts = n(),
    pats = sum(is_pat == 1L, na.rm = TRUE),
    min_distance = min(kick_distance, na.rm = TRUE),
    max_distance = max(kick_distance, na.rm = TRUE),
    missing_temp = sum(is.na(temp)),
    missing_wind = sum(is.na(wind))
  )

fg_nonattempts %>%
  summarise(
    opportunities = n(),
    min_distance = min(kick_distance, na.rm = TRUE),
    max_distance = max(kick_distance, na.rm = TRUE)
  )

game_id,season,week,kick_result,kick_distance,wind,wind_z,temp,temp_z,kicker_player_name,kick_made
<chr>,<int>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<int>
2012_01_ATL_KC,2012,1,made,20,7,0.230199,69,0.4451003,M.Bryant,1
2012_01_ATL_KC,2012,1,made,39,7,0.230199,69,0.4451003,R.Succop,1
2012_01_ATL_KC,2012,1,made,34,7,0.230199,69,0.4451003,M.Bryant,1
2012_01_ATL_KC,2012,1,made,20,7,0.230199,69,0.4451003,R.Succop,1
2012_01_ATL_KC,2012,1,made,20,7,0.230199,69,0.4451003,M.Bryant,1
2012_01_ATL_KC,2012,1,made,20,7,0.230199,69,0.4451003,R.Succop,1


game_id,season,week,posteam,defteam,kick_distance,wind,temp,ydstogo,score_differential
<chr>,<int>,<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2012_01_ATL_KC,2012,1,KC,ATL,79,7,69,9,-23
2012_01_ATL_KC,2012,1,ATL,KC,92,7,69,4,23
2012_01_BUF_NYJ,2012,1,BUF,NYJ,93,1,74,6,-14
2012_01_BUF_NYJ,2012,1,BUF,NYJ,106,1,74,19,-27
2012_01_BUF_NYJ,2012,1,NYJ,BUF,76,1,74,13,27
2012_01_BUF_NYJ,2012,1,NYJ,BUF,97,1,74,10,20


attempts,pats,min_distance,max_distance,missing_temp,missing_wind
<int>,<int>,<dbl>,<dbl>,<int>,<int>
30397,16656,18,71,0,0


opportunities,min_distance,max_distance
<int>,<dbl>,<dbl>
38780,18,116


## Persist Outputs

In [21]:
get_schema <- function(df, n = 5) {
  tibble::tibble(
    name = names(df),
    class = purrr::map_chr(df, ~ paste(class(.x), collapse = '/')),
    sample = purrr::map_chr(df, ~ paste(head(.x, n), collapse = ', '))
  )
}

attempts_path <- file.path(data_dir, 'fg_attempts.csv')
nonattempts_path <- file.path(data_dir, 'fg_nonattempts.csv')
all_path <- file.path(data_dir, 'fg_all.csv')

readr::write_csv(fg_attempts, attempts_path)
readr::write_csv(fg_nonattempts, nonattempts_path)
readr::write_csv(fg_all, all_path)

schema_preview <- get_schema(fg_all)
readr::write_csv(schema_preview, file.path(reports_dir, 'data_prep_schema_preview.csv'))
schema_preview

name,class,sample
<chr>,<chr>,<chr>
game_id,character,"2012_01_ATL_KC, 2012_01_ATL_KC, 2012_01_ATL_KC, 2012_01_ATL_KC, 2012_01_ATL_KC"
play_id,numeric,"321, 588, 727, 983, 1213"
old_game_id,numeric,"2012090908, 2012090908, 2012090908, 2012090908, 2012090908"
season,integer,"2012, 2012, 2012, 2012, 2012"
week,integer,"1, 1, 1, 1, 1"
season_type,character,"REG, REG, REG, REG, REG"
playoffs,integer,"0, 0, 0, 0, 0"
qtr,integer,"1, 1, 1, 2, 2"
game_date,Date,"2012-09-09, 2012-09-09, 2012-09-09, 2012-09-09, 2012-09-09"


In [22]:
list(
  fg_attempts = attempts_path,
  fg_nonattempts = nonattempts_path,
  fg_all = all_path,
  schema_preview = file.path(reports_dir, 'data_prep_schema_preview.csv')
)

$fg_attempts
[1] "g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/data/fg_attempts.csv"

$fg_nonattempts
[1] "g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/data/fg_nonattempts.csv"

$fg_all
[1] "g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/data/fg_all.csv"

$schema_preview
[1] "g:/Other computers/Desktop/My Files/Python/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/data_prep_schema_preview.csv"

## Session Info

In [23]:
info <- capture.output(sessionInfo())
readr::write_lines(info, file.path(reports_dir, 'session_info.txt'), append = TRUE)
cat(info, sep = '\n')

R version 4.5.1 (2025-06-13 ucrt)
Platform: x86_64-w64-mingw32/x64
Running under: Windows 11 x64 (build 26200)

Matrix products: default
  LAPACK version 3.12.1

locale:
[1] LC_COLLATE=English_United States.utf8 
[2] LC_CTYPE=English_United States.utf8   
[3] LC_MONETARY=English_United States.utf8
[4] LC_NUMERIC=C                          
[5] LC_TIME=English_United States.utf8    

time zone: America/New_York
tzcode source: internal

attached base packages:
[1] splines   stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
 [1] glue_1.8.0      nflreadr_1.5.0  lubridate_1.9.4 forcats_1.0.1  
 [5] rlang_1.1.6     yaml_2.3.10     pROC_1.19.0.1   glmmTMB_1.1.13 
 [9] mgcv_1.9-3      nlme_3.1-168    ggplot2_4.0.0   purrr_1.1.0    
[13] stringr_1.5.2   readr_2.1.5     tidyr_1.3.1     tibble_3.3.0   
[17] dplyr_1.1.4    

loaded via a namespace (and not attached):
 [1] gtable_0.3.6        TMB_1.9.18          lattice_0.22-7     
 [4] tzdb_0.5.0  